# Puxando arquivos

In [ ]:
import pandas as pd

df1 = pd.read_csv('../data/DENGBR17.csv')
df2 = pd.read_csv('../data/DENGBR18.csv')
df3 = pd.read_csv('../data/DENGBR19.csv')

df_todas = pd.concat([df1, df2, df3])
df_todas.to_parquet('../data/DENGBR_todas.parquet', index=False)

In [ ]:
df1.to_parquet

# Separando Colunas para Membro Angel Mansilla

In [ ]:
df = df_todas[['ALRM_HEPAT', 'COPAISINF', 'GRAV_METRO', 'COMUNINF', 'AUTO_IMUNE', 'LACO', 'PETEQUIAS', 'LACO_N', 'GRAV_HIPOT', 'RENAL', 'DOR_RETRO', 'CLINC_CHIK', 'NDUPLIC_N', 'CON_FHD', 'EVIDENCIA', 'ARTRALGIA', 'CEFALEIA', 'GRAV_TAQUI', 'ALRM_HEMAT', 'SEM_PRI', 'GRAV_HEMAT', 'CLASSI_FIN', 'HEMATURA', 'RESUL_VI_N', 'EXANTEMA', 'VOMITO', 'DT_NASC', 'GRAV_PULSO', 'CS_RACA', 'ALRM_PLAQ', 'DT_ALRM', 'GRAV_SANG', 'PLASMATICO', 'PETEQUIA_N', 'CS_GESTANT', 'GRAV_AST', 'GRAV_ENCH', 'GRAV_MIOC', 'GRAV_CONV']]

# Limpando Colunas

In [ ]:
# Removes columns that do data leakage
cols_leakage = [
    "CLASSI_FIN",
    "CRITERIO",
    "DT_ENCERRA",

    "ALRM_HIPOT",
    "ALRM_PLAQ",
    "ALRM_VOM",
    "ALRM_SANG",
    "ALRM_HEMAT",
    "ALRM_ABDOM",
    "ALRM_LETAR",
    "ALRM_HEPAT",
    "ALRM_LIQ",
    "DT_ALRM",

    "GRAV_PULSO",
    "GRAV_CONV",
    "GRAV_ENCH",
    "GRAV_INSUF",
    "GRAV_TAQUI",
    "GRAV_EXTRE",
    "GRAV_HIPOT",
    "GRAV_HEMAT",
    "GRAV_MELEN",
    "GRAV_METRO",
    "GRAV_SANG",
    "GRAV_AST",
    "GRAV_MIOC",
    "GRAV_CONSC",
    "GRAV_ORGAO",
    "DT_GRAV",
    "COPAISINF",
    "COMUNINF"
]

df = df.drop(cols_leakage, axis=1, errors='ignore')

In [ ]:
# Removes columns that have all null values or only one unique value
def remove_same_val(df):
    for column in df.columns:
        if df[column].isnull().all():
            print(f"Column '{column}' has all null values.")
            df = df.drop(column, axis=1)
        elif df[column].nunique() == 1:
            print(f"Column '{column}' has only one unique value: {df[column].unique()[0]}")
            df = df.drop(column, axis=1)
    return df

df = remove_same_val(df)

In [ ]:
# Creates columns based on the DT_NASC
df['DT_NASC'] = pd.to_datetime(df['DT_NASC'], errors='coerce')
df['NASC_YEAR'] = df['DT_NASC'].dt.year
df['AGE'] = 2020 - df['NASC_YEAR']

df = df.drop(['DT_NASC', 'NASC_YEAR'], axis=1, errors='ignore')

In [ ]:
# Removes columns related to comorbidities beacuase they don´t have relation to predicting dengue
cols_comorbidities = ['AUTO_IMUNE', 'RENAL']
df = df.drop(cols_comorbidities, axis=1, errors='ignore')

In [ ]:
# Columns that were judged to be not relevant for predicting dengue
cols_irrelevant = ['CS_GESTANT', 'CS_RACA', 'CLINC_CHIK', 'RESUL_VI_N']
df = df.drop(cols_irrelevant, axis=1, errors='ignore')

In [ ]:
# Separating the SEM_PRI column into two columns: sem_pri_year and sem_pri_week
def split_sem_pri(df):
    df["SEM_PRI_YEAR"] = df["SEM_PRI"] // 100
    df["SEM_PRI_WEEK"] = df["SEM_PRI"] % 100
    df = df.drop(columns=["SEM_PRI"])
    return df

df = split_sem_pri(df)

# Resultado Final após Colunas Limpas

In [ ]:
print(list(df.columns))
display(df)